# 정수 인코딩과 패딩

### 필요한 패키지 설치

영어 원문을 토큰화한 뒤 정수 시퀀스로 변환하려면 다음 두 패키지를 사용한다.

- **NLTK**: 문장·단어 토큰화와 불용어 목록을 제공하는 자연어 처리 라이브러리
- **TensorFlow Keras**: 신경망 구성과 텍스트 전처리 도구를 제공하는 딥러닝 API


In [21]:
%pip install -q nltk tensorflow


Note: you may need to restart the kernel to use updated packages.


## 정수 인코딩과 패딩

### 단어 집합과 정수 인코딩

**정수 인코딩**은 먼저 모델이 사용할 단어를 정하고 각 단어에 구별 가능한 번호를 배정하는 흐름이다.

- **단어 집합(Vocabulary)**: 모델이 구분할 토큰과 정수 ID의 대응 목록
- **정수 ID**: 토큰을 구분하기 위한 식별자이며 숫자 크기는 중요도나 의미와 무관
- **단어-정수 사전**: `{'little': 3, 'rose': 5}`처럼 토큰을 정수 ID로 찾기 위한 매핑

모델은 문자열을 직접 계산하지 못하므로 `little → 3`, `rose → 5`와 같이 토큰을 ID로 바꾼다. 이때 학습에 사용할 단어 수를 제한하면 사전에 없는 단어와 길이가 다른 문장을 처리할 별도 규칙이 필요하다.

### PAD와 OOV가 필요한 이유

정수 인코딩에서는 다음 두 문제가 발생한다.

1. 단어 집합에 없는 토큰이 입력될 수 있다.
2. 문장마다 토큰 수가 달라 하나의 2차원 배열로 묶기 어렵다.

- **OOV(Out-Of-Vocabulary)**: 미등록 토큰을 하나의 ID로 바꿔 문장 안의 위치와 길이를 유지하는 특수 토큰
- **PAD(Padding Token)**: 실제 단어가 아닌, 짧은 문장의 빈 위치를 채워 길이를 맞추는 자리 표시자
- **특수 ID 예약**: PAD와 OOV의 ID가 일반 단어 ID와 겹치지 않도록 특정 번호를 미리 비워 두는 과정

이번에는 PAD에 `0`, OOV에 `1`, 일반 단어에 `2` 이상의 ID를 사용한다. `0`과 `1`은 반드시 지켜야 하는 법칙이 아니라 널리 사용하는 관례이다.
다른 번호를 사용해도 되지만 PAD·OOV·일반 단어가 서로 겹치지 않고 전처리와 모델 전체에서 같은 규칙을 사용해야 한다.

PAD에 `0`을 자주 사용하는 이유는 `np.zeros()`로 패딩 배열을 바로 만들 수 있고, 마스크나 임베딩 계층에서 0번 위치를 패딩으로 처리하기 편하기 때문이다. OOV는 그다음 비어 있는 `1`을 사용하는 경우가 많다.

예를 들어 `little=3`, `rose=5`인 단어 집합에서 `['little', 'dragon', 'rose']`는 `[3, X, 5]`가 된다.
이때, `dragon`이 미등록 단어라고 하면 OOV `1`로 바꿔 `[3, 1, 5]가 되고`,
길이가 5로 설정되고 앞쪽을 패딩하면 `[0, 0, 3, 1, 5]`가 된다.

### 패딩과 잘림

문장을 정수로 바꾼 뒤에는 여러 문장을 하나의 배열로 묶을 수 있도록 길이를 맞춰야 한다.

- **PAD ID `0`**: 실제 토큰이 없는 위치를 표시하며 마스크나 `padding_idx`로 학습에서 제외하는 정수 값
- **잘림(Truncation)**: `maxlen`보다 긴 시퀀스에서 일부 토큰을 삭제하는 과정
- **`pre` / `post`**: 각각 시퀀스의 앞쪽과 뒤쪽을 나타내는 방향 설정

패딩은 PAD 위치만 바꾸지만 잘림은 실제 토큰을 삭제한다. 따라서 잘림 방향은 중요한 정보가 있는 위치를 고려해 선택한다.


### NLTK 언어 자원 다운로드

단어 집합을 만들기 전에 원문을 문장과 단어로 나누고 불필요한 단어를 걸러낼 언어 자원을 준비한다.

- **`nltk.download()`**: 이름으로 지정한 NLTK 언어 자원을 내려받는 함수
- **`punkt` / `punkt_tab`**: 영어 문장과 단어의 경계를 판별할 때 사용하는 토큰화 자원
- **`stopwords`**: 영어에서 자주 등장하지만 분석 목적에 따라 제거할 수 있는 불용어 목록


In [22]:
import nltk

nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\playdata2\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\playdata2\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\playdata2\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

### 어린왕자 영어 원문 준비

반복되는 단어와 길이가 다른 문장을 가진 영어 원문을 준비한다. 단어 반복은 빈도 기반 ID 배정에, 문장 길이 차이는 패딩 비교에 사용한다.


In [23]:
raw_text = """The Little Prince, written by Antoine de Saint-Exupéry, is a poetic tale about a young prince who travels from his home planet to Earth. The story begins with a pilot stranded in the Sahara Desert after his plane crashes. While trying to fix his plane, he meets a mysterious young boy, the Little Prince.

The Little Prince comes from a small asteroid called B-612, where he lives alone with a rose that he loves deeply. He recounts his journey to the pilot, describing his visits to several other planets. Each planet is inhabited by a different character, such as a king, a vain man, a drunkard, a businessman, a geographer, and a fox. Through these encounters, the Prince learns valuable lessons about love, responsibility, and the nature of adult behavior.

On Earth, the Little Prince meets various creatures, including a fox, who teaches him about relationships and the importance of taming, which means building ties with others. The fox's famous line, "You become responsible, forever, for what you have tamed," resonates with the Prince's feelings for his rose.

Ultimately, the Little Prince realizes that the essence of life is often invisible and can only be seen with the heart. After sharing his wisdom with the pilot, he prepares to return to his asteroid and his beloved rose. The story concludes with the pilot reflecting on the lessons learned from the Little Prince and the enduring impact of their friendship.

The narrative is a beautifully simple yet profound exploration of love, loss, and the importance of seeing beyond the surface of things."""


## 토큰과 빈도 준비

정수 인코딩 전 원문을 문장별 토큰 목록으로 바꾸고 전체 토큰의 등장 횟수를 계산한다.

- **`sent_tokenize()`**: 문서 문자열을 문장 문자열 목록으로 나누는 함수
- **`word_tokenize()`**: 문장 문자열을 단어와 구두점 토큰으로 나누는 함수
- **`stopwords.words('english')`**: NLTK의 영어 불용어 목록을 반환하는 함수
- **`lower()`**: 대소문자 차이를 줄이도록 문자열의 영문 대문자를 소문자로 바꾸는 메서드
- **`preprocessed_sentences`**: 문장별 토큰 목록을 저장하는 2중 리스트
- **`vocab`**: `{토큰: 등장 횟수}` 형태로 빈도를 저장하는 딕셔너리

문장별 토큰 목록은 이후 정수 인코딩의 입력이 되고, 누적 빈도는 단어별 ID 순서를 정하는 기준이 된다.


In [24]:
from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.corpus import stopwords

sentences = sent_tokenize(raw_text)  # 문장 토큰화
en_stopwords = stopwords.words('english')  # 영어 불용어 가져오기

vocab = {}  # {토큰: 등장 횟수} 형태의 토큰 사전
preprocessed_sentences = []

for sentence in sentences:
    sentence = sentence.lower()  # 문장 전체를 소문자로 변경
    tokens = word_tokenize(sentence)

    tokens = [token for token in tokens if token not in en_stopwords and len(token) > 2]

    # 추출된 토큰 목록을 list에 저장
    preprocessed_sentences.append(tokens)

    # 동일한 토큰의 등장 횟수 누적
    for token in tokens:
        if token not in vocab:
            vocab[token] = 0
        vocab[token] += 1

for sentence_number, tokens in enumerate(preprocessed_sentences, start=1):
    print(f'{sentence_number}: {tokens}')


1: ['little', 'prince', 'written', 'antoine', 'saint-exupéry', 'poetic', 'tale', 'young', 'prince', 'travels', 'home', 'planet', 'earth']
2: ['story', 'begins', 'pilot', 'stranded', 'sahara', 'desert', 'plane', 'crashes']
3: ['trying', 'fix', 'plane', 'meets', 'mysterious', 'young', 'boy', 'little', 'prince']
4: ['little', 'prince', 'comes', 'small', 'asteroid', 'called', 'b-612', 'lives', 'alone', 'rose', 'loves', 'deeply']
5: ['recounts', 'journey', 'pilot', 'describing', 'visits', 'several', 'planets']
6: ['planet', 'inhabited', 'different', 'character', 'king', 'vain', 'man', 'drunkard', 'businessman', 'geographer', 'fox']
7: ['encounters', 'prince', 'learns', 'valuable', 'lessons', 'love', 'responsibility', 'nature', 'adult', 'behavior']
8: ['earth', 'little', 'prince', 'meets', 'various', 'creatures', 'including', 'fox', 'teaches', 'relationships', 'importance', 'taming', 'means', 'building', 'ties', 'others']
9: ['fox', 'famous', 'line', 'become', 'responsible', 'forever', 'tame

### 토큰 빈도 확인
전체 토큰 종류의 수와 `prince`, `little`의 빈도를 함께 확인한다.


In [25]:
print('단어 집합 후보 수:', len(vocab))
print('prince 빈도:', vocab['prince'])
print('little 빈도:', vocab['little'])
vocab


단어 집합 후보 수: 109
prince 빈도: 9
little 빈도: 6


{'little': 6,
 'prince': 9,
 'written': 1,
 'antoine': 1,
 'saint-exupéry': 1,
 'poetic': 1,
 'tale': 1,
 'young': 2,
 'travels': 1,
 'home': 1,
 'planet': 2,
 'earth': 2,
 'story': 2,
 'begins': 1,
 'pilot': 4,
 'stranded': 1,
 'sahara': 1,
 'desert': 1,
 'plane': 2,
 'crashes': 1,
 'trying': 1,
 'fix': 1,
 'meets': 2,
 'mysterious': 1,
 'boy': 1,
 'comes': 1,
 'small': 1,
 'asteroid': 2,
 'called': 1,
 'b-612': 1,
 'lives': 1,
 'alone': 1,
 'rose': 3,
 'loves': 1,
 'deeply': 1,
 'recounts': 1,
 'journey': 1,
 'describing': 1,
 'visits': 1,
 'several': 1,
 'planets': 1,
 'inhabited': 1,
 'different': 1,
 'character': 1,
 'king': 1,
 'vain': 1,
 'man': 1,
 'drunkard': 1,
 'businessman': 1,
 'geographer': 1,
 'fox': 3,
 'encounters': 1,
 'learns': 1,
 'valuable': 1,
 'lessons': 2,
 'love': 2,
 'responsibility': 1,
 'nature': 1,
 'adult': 1,
 'behavior': 1,
 'various': 1,
 'creatures': 1,
 'including': 1,
 'teaches': 1,
 'relationships': 1,
 'importance': 2,
 'taming': 1,
 'means': 1,
 '

## 빈도순 단어 집합 만들기

토큰과 빈도 쌍을 빈도 내림차순으로 정렬해 높은 빈도의 토큰부터 낮은 ID를 부여할 준비를 한다.

- **`sorted()`**: 반복 가능한 데이터를 지정한 기준에 따라 정렬한 새 리스트로 반환하는 함수
- **`key=lambda item: item[1]`**: `(토큰, 빈도)` 중 두 번째 값인 빈도를 정렬 기준으로 지정

In [26]:
import pandas as pd

vocab_sorted = sorted(vocab.items(), key=lambda item: item[1], reverse=True)
vocab_df = pd.DataFrame(vocab_sorted, columns=['word', 'freq'])
vocab_df


,word,freq
0,prince,9
1,little,6
2,pilot,4
3,rose,3
4,fox,3
...,...,...
104,loss,1
105,seeing,1
106,beyond,1
107,surface,1


### PAD와 OOV를 예약한 단어-정수 사전

- **OOV(Out-Of-Vocabulary)**: 미등록 토큰을 하나의 ID로 바꿔 문장 안의 위치와 길이를 유지하는 특수 토큰
- **PAD(Padding Token)**: 실제 단어가 아닌, 짧은 문장의 빈 위치를 채워 길이를 맞추는 자리 표시자

여기서 **예약한다**는 말은 PAD와 OOV를 실제 문장에 추가한다는 뜻이 아니라, 일반 단어가 ID `0`과 `1`을 사용하지 못하도록 자리를 비워 둔다는 뜻이다. 특수 ID를 먼저 정해야 뒤에서 일반 단어와 번호가 충돌하지 않는다.

- **`word_to_index`**: 일반 단어를 정수 ID로 바꾸기 위한 `{단어: ID}` 사전
- **`i + 2`**: 일반 단어 ID가 `2`부터 시작하도록 `enumerate()` 순번을 이동하는 식
- **`max_token_id = 15`**: 일반 단어 14개에 ID `2~15`만 사용하도록 정한 마지막 ID

실제 프로젝트에서는 데이터 크기와 메모리, OOV 비율을 고려해 단어 집합 크기를 정한다.


In [27]:
word_to_index = {
    word_and_count[0]: i + 2
    for i, word_and_count in enumerate(vocab_sorted)
}

max_token_id = 15

# ID가 15 이하인 빈도수 상위 단어만 목록에 남김
word_to_index = {
    word: token_id
    for word, token_id in word_to_index.items()
    if token_id <= max_token_id
}
word_to_index

{'prince': 2,
 'little': 3,
 'pilot': 4,
 'rose': 5,
 'fox': 6,
 'young': 7,
 'planet': 8,
 'earth': 9,
 'story': 10,
 'plane': 11,
 'meets': 12,
 'asteroid': 13,
 'lessons': 14,
 'love': 15}

### OOV ID 추가

앞 셀에서는 OOV가 사용할 ID `1`을 비워 두기만 했다. 이제 정수 인코딩 중 사전에 없는 토큰을 만났을 때 반환할 실제 기본값을 등록한다.

- **`OOV_TOKEN = 'OOV'`**: 사람이 사전에서 용도를 알아볼 수 있게 붙인 특수 토큰 이름
- **`OOV_ID = 1`**: 모든 미등록 토큰이 공통으로 사용할 정수 ID

PAD는 원문 토큰을 ID로 바꾸는 단계에서 등장하지 않고, 문장 길이를 맞추는 패딩 단계에서 삽입된다. 따라서 현재 `word_to_index`에는 OOV만 추가하고 PAD `0`은 계속 비워 둔다.


In [28]:
OOV_TOKEN = 'OOV'
OOV_ID = 1
word_to_index[OOV_TOKEN] = OOV_ID
word_to_index


{'prince': 2,
 'little': 3,
 'pilot': 4,
 'rose': 5,
 'fox': 6,
 'young': 7,
 'planet': 8,
 'earth': 9,
 'story': 10,
 'plane': 11,
 'meets': 12,
 'asteroid': 13,
 'lessons': 14,
 'love': 15,
 'OOV': 1}

### 정수 ID를 다시 단어로 해석하기

정수 인코딩이 끝나면 모델은 `[3, 1, 5]`처럼 숫자만 입력받는다. 모델 학습에는 숫자만 있으면 되지만, 사람이 전처리 결과를 확인하거나 예측 결과를 설명하려면 각 ID가 어떤 토큰인지 다시 읽을 수 있어야 한다.

- **정방향 사전 `word_to_index`**: `{'little': 3}`처럼 토큰을 ID로 바꾸며 모델 입력을 만들 때 사용
- **역방향 사전 `index_to_word`**: `{3: 'little'}`처럼 ID를 토큰으로 바꾸며 결과 확인과 디버깅에 사용

역방향 사전은 모델 학습에 반드시 필요한 구조가 아니라 사람이 숫자 시퀀스를 해석하기 위한 보조 도구이다. 또한 여러 미등록 단어가 모두 OOV `1`로 합쳐지므로 `1 → OOV`까지는 확인할 수 있지만 원래 단어가 `dragon`이었는지는 복원할 수 없다.

PAD `0`은 `word_to_index`에 등록하지 않았으므로 현재 역방향 사전에도 없다. 패딩된 시퀀스를 읽을 때는 ID `0`을 PAD로 별도 처리해야 한다.


In [29]:
index_to_word = {
    token_id: word
    for word, token_id in word_to_index.items()
}
index_to_word


{2: 'prince',
 3: 'little',
 4: 'pilot',
 5: 'rose',
 6: 'fox',
 7: 'young',
 8: 'planet',
 9: 'earth',
 10: 'story',
 11: 'plane',
 12: 'meets',
 13: 'asteroid',
 14: 'lessons',
 15: 'love',
 1: 'OOV'}

## 문장별 정수 인코딩

이제 문장마다 저장한 토큰 목록을 모델이 받을 수 있는 정수 ID 목록으로 바꾼다. 토큰의 순서는 그대로 유지하며, 사전에 있는 단어는 고유 ID로, 사전에 없는 단어는 OOV `1`로 치환한다.

- **`dict.get(key, default)`**: 키가 사전에 있으면 대응 값을 반환하고, 없으면 오류 대신 `default`를 반환하는 메서드
- **`word_to_index.get(token, OOV_ID)`**: 등록 토큰은 고유 ID로, 미등록 토큰은 OOV ID `1`로 변환

OOV를 사용하지 않고 `word_to_index[token]`으로만 조회하면 미등록 단어에서 `KeyError`가 발생한다. 미등록 단어를 삭제하면 문장 길이와 위치가 달라지지만 OOV로 바꾸면 해당 위치를 유지할 수 있다.

출력에서는 첫 문장의 토큰과 ID를 같은 위치끼리 비교한다. `little → 3`, `prince → 2`처럼 등록 단어는 고유 ID를 사용하고, 제한된 단어 집합에 없는 단어는 `1`로 나타나야 한다.


In [30]:
encoded_sentences = []  # 정수 인코딩된 문장 저장용 list

# 토큰 순서를 유지하면서 토큰(단어)를 토큰 ID로 변환
# 단, 미등록된 토큰(단어)은 OOV 값 1로 변환
for tokens in preprocessed_sentences:

    token_ids = [
        # 단어-정수 사전에서 token과 일치하는 정수(토큰ID) 얻어오기
        # 단, 없으면 기본값으로 OOV_ID (1) 사용
        word_to_index.get(token, OOV_ID)
        for token in tokens
    ]

    encoded_sentences.append(token_ids)
    print(token_ids)

print("문장별 길이: ", [
    len(token_ids)
    for token_ids in encoded_sentences
])

[3, 2, 1, 1, 1, 1, 1, 7, 2, 1, 1, 8, 9]
[10, 1, 4, 1, 1, 1, 11, 1]
[1, 1, 11, 12, 1, 7, 1, 3, 2]
[3, 2, 1, 1, 13, 1, 1, 1, 1, 5, 1, 1]
[1, 1, 4, 1, 1, 1, 1]
[8, 1, 1, 1, 1, 1, 1, 1, 1, 1, 6]
[1, 2, 1, 1, 14, 15, 1, 1, 1, 1]
[9, 3, 2, 12, 1, 1, 1, 6, 1, 1, 1, 1, 1, 1, 1, 1]
[6, 1, 1, 1, 1, 1, 1, 1, 2, 1, 5]
[1, 3, 2, 1, 1, 1, 1, 1, 1, 1]
[1, 1, 4, 1, 1, 13, 1, 5]
[10, 1, 4, 1, 14, 1, 3, 2, 1, 1, 1]
[1, 1, 1, 1, 1, 1, 15, 1, 1, 1, 1, 1, 1]
문장별 길이:  [13, 8, 9, 12, 7, 11, 10, 16, 11, 10, 8, 11, 13]


### 새로운 문장의 OOV 처리

학습 데이터로 만든 단어 집합을 실제 새 문장에 적용하면 처음 보는 단어가 나타날 수 있다.
등록 단어와 미등록 단어가 섞인 짧은 문장으로 OOV 규칙과 역방향 사전의 한계를 함께 확인한다.

`little`과 `rose`는 각각 기존 ID `3`, `5`를 유지하고 `dragon`은 OOV `1`이 된다. 역방향 변환 결과는 `['little', 'OOV', 'rose']`이며, OOV로 합쳐진 뒤에는 원래 단어가 무엇이었는지 복원할 수 없다.


In [31]:
new_tokens = ['little', 'dragon', 'rose']
new_token_ids = [word_to_index.get(token, OOV_ID) for token in new_tokens]
decoded_tokens = [index_to_word.get(token_id, 'PAD') for token_id in new_token_ids]

print('새 문장 토큰:', new_tokens)
print('새 문장 ID:', new_token_ids)
print('역변환 결과:', decoded_tokens)


새 문장 토큰: ['little', 'dragon', 'rose']
새 문장 ID: [3, 1, 5]
역변환 결과: ['little', 'OOV', 'rose']


## NumPy로 패딩 원리 구현

정수 인코딩이 끝나도 문장 길이는 7~16개로 서로 다르다.
여러 문장을 하나의 배치로 묶으려면 모든 행의 열 수가 같아야 하므로, 가장 긴 문장 길이에 맞춘 2차원 배열을 만들고 빈 위치를 PAD `0`으로 채운다.

- **`np.zeros(shape, dtype)`**: 지정한 shape를 `0`으로 채워 빈 패딩 틀을 만드는 함수

원래 ID를 행의 앞에 복사하면 오른쪽에 PAD가 남는 `post` 패딩이 되고,
행의 뒤에 맞춰 복사하면 왼쪽에 PAD가 남는 `pre` 패딩이 된다.
PAD는 길이를 맞추기 위한 값이므로 이후 모델에서는 실제 단어처럼 학습되지 않도록 마스크 또는 `padding_idx`로 구분해야 한다.


In [32]:
import numpy as np

# 토큰 수가 가장 많은 문장의 토큰 개수 구하기
max_len = max([
    len(token_ids)
    for token_ids in encoded_sentences
])  # 16

batch_size = len(encoded_sentences)  # 13 문장 == 13 행

# 0으로 채워진 13행 16열 짜리 ndarray 생성
# 왜 13형? 원문의 문장 토큰화 -> 13 문장
# 왜 16열? 문장의 단어 토큰화 -> 제일 토큰이 많은 문장의 토큰 수가 16개

# 토큰 수가 모자라는 행은 뒤에 PAD 값 0을 추가하여 만든 배열
post_padded = np.zeros((batch_size, max_len), dtype=int)

# 토큰 수가 모자라는 행은 앞에 PAD 값 0을 추가하여 만든 배열
pre_padded = np.zeros((batch_size, max_len), dtype=int)

for i, token_ids in enumerate(encoded_sentences):

    post_padded[i, :len(token_ids)] = token_ids  # 앞에서 부터 채우기
    pre_padded[i, -len(token_ids):] = token_ids  # 뒤에서 부터 채우기

# post_padded
pre_padded

array([[ 0,  0,  0,  3,  2,  1,  1,  1,  1,  1,  7,  2,  1,  1,  8,  9],
       [ 0,  0,  0,  0,  0,  0,  0,  0, 10,  1,  4,  1,  1,  1, 11,  1],
       [ 0,  0,  0,  0,  0,  0,  0,  1,  1, 11, 12,  1,  7,  1,  3,  2],
       [ 0,  0,  0,  0,  3,  2,  1,  1, 13,  1,  1,  1,  1,  5,  1,  1],
       [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  1,  1,  4,  1,  1,  1,  1],
       [ 0,  0,  0,  0,  0,  8,  1,  1,  1,  1,  1,  1,  1,  1,  1,  6],
       [ 0,  0,  0,  0,  0,  0,  1,  2,  1,  1, 14, 15,  1,  1,  1,  1],
       [ 9,  3,  2, 12,  1,  1,  1,  6,  1,  1,  1,  1,  1,  1,  1,  1],
       [ 0,  0,  0,  0,  0,  6,  1,  1,  1,  1,  1,  1,  1,  2,  1,  5],
       [ 0,  0,  0,  0,  0,  0,  1,  3,  2,  1,  1,  1,  1,  1,  1,  1],
       [ 0,  0,  0,  0,  0,  0,  0,  0,  1,  1,  4,  1,  1, 13,  1,  5],
       [ 0,  0,  0,  0,  0, 10,  1,  4,  1, 14,  1,  3,  2,  1,  1,  1],
       [ 0,  0,  0,  1,  1,  1,  1,  1,  1, 15,  1,  1,  1,  1,  1,  1]])

## Keras로 단어 집합과 정수 ID 만들기

- **Keras**: 신경망 모델을 간결하게 설계하고 학습할 수 있도록 TensorFlow가 제공하는 고수준 딥러닝 API

여기서는 정수 인코딩과 패딩 결과를 비교하는 유틸리티로 사용한다.

앞에서는 특수 ID 예약, 단어-ID 매핑과 OOV 치환을 직접 구현했다. 여기서는 같은 과정을 Keras `Tokenizer`가 어떤 규칙으로 처리하는지 비교한다.

- **`Tokenizer`**: 텍스트의 토큰 빈도를 학습해 단어-ID 사전을 만들고 문장을 정수 시퀀스로 변환하는 클래스
- **`num_words=16`**: 실제 변환 범위를 ID `1~15`로 제한해 일반 단어에 `2~15`를 사용하는 설정
- **`oov_token='OOV'`**: 미등록 단어를 대신할 토큰을 지정하고 OOV에 ID `1`을 배정하는 설정
- **`fit_on_texts()`**: 입력 텍스트의 빈도와 등장 순서를 분석해 `word_index`를 만드는 메서드

Keras `Tokenizer`도 ID `0`을 단어에 배정하지 않고 패딩을 위해 비워 둔다. 이는 Keras 전처리 도구의 기본 관례이며, 모든 NLP 모델에서 PAD와 OOV를 반드시 각각 `0`, `1`로 지정할 필요는 없다.


In [33]:
from tensorflow.keras.preprocessing.text import Tokenizer

tokenizer = Tokenizer(num_words=max_token_id + 1, oov_token=OOV_TOKEN)
tokenizer.fit_on_texts(preprocessed_sentences)


### Keras의 단어-ID 매핑

학습을 마친 `Tokenizer`에서 전체 사전과 실제 변환에 사용할 ID 범위를 나누어 확인한다.

- **`word_index`**: `fit_on_texts()`가 만든 `{단어: 정수 ID}` 형태의 전체 사전
- **`num_words` 적용 범위**: `word_index`에는 제한 밖의 단어도 남지만 실제 정수 변환에는 지정 범위의 ID만 사용

OOV `1`, prince `2`, little `3`의 매핑과 실제 변환 범위를 확인한다.


In [34]:
print('전체 word_index 항목 수:', len(tokenizer.word_index))
active_word_index = {
    word: token_id
    for word, token_id in tokenizer.word_index.items()
    if token_id <= max_token_id
}
print('변환에 사용하는 매핑:', active_word_index)


전체 word_index 항목 수: 110
변환에 사용하는 매핑: {'OOV': 1, 'prince': 2, 'little': 3, 'pilot': 4, 'rose': 5, 'fox': 6, 'young': 7, 'planet': 8, 'earth': 9, 'story': 10, 'plane': 11, 'meets': 12, 'asteroid': 13, 'lessons': 14, 'love': 15}


### Keras 정수 시퀀스 변환

단어-ID 매핑을 확인했으면 같은 토큰 목록을 Keras 방식의 정수 시퀀스로 변환한다.

- **`texts_to_sequences()`**: 학습된 `word_index`를 사용해 각 문장의 토큰을 정수 ID 목록으로 변환하는 메서드

변환 결과는 아직 문장마다 길이가 다르다. 직접 구현한 정수 시퀀스와의 비교값이 `True`인지 확인한다.


In [35]:
keras_encoded_sentences = tokenizer.texts_to_sequences(preprocessed_sentences)

print('첫 문장 ID:', keras_encoded_sentences[0])
print('문장별 길이:', [len(token_ids) for token_ids in keras_encoded_sentences])
print('직접 구현과 동일:', keras_encoded_sentences == encoded_sentences)


첫 문장 ID: [3, 2, 1, 1, 1, 1, 1, 7, 2, 1, 1, 8, 9]
문장별 길이: [13, 8, 9, 12, 7, 11, 10, 16, 11, 10, 8, 11, 13]
직접 구현과 동일: True


### `pre` 패딩과 `post` 패딩

길이가 다른 정수 시퀀스를 모델에 한 번에 전달하려면 동일한 열 수를 가진 2차원 배열로 맞춰야 한다.

- **`pad_sequences()`**: 길이가 다른 정수 시퀀스를 같은 길이의 2차원 배열로 만드는 함수
- **`padding='pre'`**: 짧은 시퀀스의 왼쪽에 PAD를 채움
- **`padding='post'`**: 짧은 시퀀스의 오른쪽에 PAD를 채움
- **`value=0`**: 빈 위치에 채울 PAD 값을 `0`으로 지정

두 결과 모두 shape은 `(문장 수, maxlen)`이며 실제 토큰 ID의 순서는 유지된다.


In [36]:
from tensorflow.keras.utils import pad_sequences

keras_post_padded = pad_sequences(keras_encoded_sentences, padding='post', value=0)
keras_pre_padded = pad_sequences(keras_encoded_sentences, padding='pre', value=0)

print('post shape:', keras_post_padded.shape)
print('post 두 번째 문장:', keras_post_padded[1].tolist())
print('pre 두 번째 문장:', keras_pre_padded[1].tolist())
print('NumPy post와 동일:', np.array_equal(keras_post_padded, post_padded))

post shape: (13, 16)
post 두 번째 문장: [10, 1, 4, 1, 1, 1, 11, 1, 0, 0, 0, 0, 0, 0, 0, 0]
pre 두 번째 문장: [0, 0, 0, 0, 0, 0, 0, 0, 10, 1, 4, 1, 1, 1, 11, 1]
NumPy post와 동일: True


### `pre` 잘림과 `post` 잘림

최대 길이를 직접 지정하면 짧은 문장은 패딩되고 긴 문장은 초과한 토큰이 잘린다.

- **`maxlen=13`**: 패딩 또는 잘림을 적용한 뒤 유지할 최종 시퀀스 길이
- **`truncating='pre'`**: 길이를 초과한 토큰을 시퀀스 앞쪽에서 제거
- **`truncating='post'`**: 길이를 초과한 토큰을 시퀀스 뒤쪽에서 제거

길이 16인 여덟 번째 문장에서 어느 세 토큰이 사라지는지 비교한다.


In [37]:
max_sequence_length = 13
pre_truncated = pad_sequences(
    keras_encoded_sentences,
    maxlen=max_sequence_length,
    padding='post',
    truncating='pre',
    value=0,
)
post_truncated = pad_sequences(
    keras_encoded_sentences,
    maxlen=max_sequence_length,
    padding='post',
    truncating='post',
    value=0,
)

print('원본 길이:', len(keras_encoded_sentences[7]))
print('원본 ID:', keras_encoded_sentences[7])
print('pre 잘림:', pre_truncated[7].tolist())
print('post 잘림:', post_truncated[7].tolist())


원본 길이: 16
원본 ID: [9, 3, 2, 12, 1, 1, 1, 6, 1, 1, 1, 1, 1, 1, 1, 1]
pre 잘림: [12, 1, 1, 1, 6, 1, 1, 1, 1, 1, 1, 1, 1]
post 잘림: [9, 3, 2, 12, 1, 1, 1, 6, 1, 1, 1, 1, 1]
